- ingest data from volume to delta table
- add metadata column such as file_name , ingest_timestamp

In [0]:
%run ../00-common/01.environment_config

In [0]:
from pyspark.sql.types import StructField,StructType,DoubleType,StringType,IntegerType,DateType
# for production scenarion we define schema explicitly if new data is also come then also no issue occur, for development purpose we can used .option("inferSchema","true") this option
 
races_schema = StructType([
    StructField("season", IntegerType()),
    StructField("round", IntegerType()),
    StructField("url", StringType()),
    StructField("raceName", DoubleType()),
    StructField("date", DateType()),
    StructField("circuitId", StringType())
])

In [0]:
df_races = spark.read.format("csv") \
            .option("header","true") \
            .schema(races_schema) \
            .load(source_file_path)


In [0]:
from pyspark.sql import functions as F

df_races_final = df_races.withColumn("ingestion_date", F.current_timestamp()) \
                            .withColumn("source_file",F.col("_metadata.file_path"))

display(df_races_final)


In [0]:
df_races_final.write.format("delta") \
                        .mode("overwrite") \
                        .saveAsTable(full_table_name)

In [0]:
%sql
select * from formula1.bronze.races;